# Study 919 — Methodology Shock 📐

**When an index changes its own rules, can you front-run the trade every tracker must make?**

An index is a rulebook. Every fund that tracks it is contractually obliged to hold whatever
the rulebook says — so when the *rules themselves* change (a special rebalance to cap
concentration, a switch to float-adjusted weights, a new eligibility filter, a change to how
constituents migrate), trillions of dollars of passive money must trade a large basket on a
publicly announced date. The folklore says: buy the affected wrapper against an unaffected
sibling between the announcement and the effective date.

We test it on **8 hardcoded index rule changes** — Nasdaq-100 special
rebalances, the S&P float-adjustment phases, the S&P multiple-share-class ban and its
reversal, Russell banding and the move to semi-annual reconstitution — with
**QQQ vs SPY**, **SPY vs IWM** and **IWM vs MDY** as treated/sibling pairs, on daily
total-return closes, 1993-01-29 → 2026-06-30 (8,411 sessions).

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `547814cd71c7`); the
live cells run the fast offline synthetic control and are labelled as such. As-of
2026-06-30.*


## 1. Why anyone would expect this to work

The mechanism is not folklore — it is a contract. If S&P announces that the index will be reweighted on a Friday, every S&P tracker on earth **must** trade that reweighting, and everybody knows it weeks in advance. Forced buyers on a known date is about as clean a set-up as markets offer.

The catch is *where* the forced trade lands. A special rebalance that halves one company's weight is enormous **for that company** — and gets diluted across the other four hundred names in the wrapper you can actually buy. This study asks only the version a price-taker can act on: does the shock survive at the **wrapper** level, after you hedge out the market and pay the spread?

> 🔬 *For the quants:* the constituent-level demand-curve literature (Shleifer 1986, Harris & Gurel 1986, Greenwood 2005) is strong and we do not dispute it. The wrapper-level residual is a different, much smaller quantity.

## 2. How we measure a 'shock'

For each rule change we take the affected fund and a sibling the change does *not* touch — QQQ against SPY for Nasdaq rules, SPY against IWM for S&P rules, IWM against MDY for Russell rules. We learn how the two normally move together over the year *before* the announcement, then ask how much the affected fund moved **beyond** what the sibling explains, over the ten sessions after the news. That excess is the shock.

One rule we never bend: the announcement is public at the closing bell of day zero, so our trade starts the *next* day. We never earn the return of the day the news broke.

In [1]:
R = dict(ann_car=-14.1, ann_p=0.802, eff_car=35.7, eff_p=0.541,
         ann_hit=43, eff_hit=57, n_events=7)
print('announcement day  : average excess move %+.1f bps   (chance of seeing this by luck: %.0f%%)'
      % (R['ann_car'], R['ann_p']*100))
print('effective day     : average excess move %+.1f bps   (chance of seeing this by luck: %.0f%%)'
      % (R['eff_car'], R['eff_p']*100))
print('the two legs point in OPPOSITE directions, on %d events each' % R['n_events'])

announcement day  : average excess move -14.1 bps   (chance of seeing this by luck: 80%)
effective day     : average excess move +35.7 bps   (chance of seeing this by luck: 54%)
the two legs point in OPPOSITE directions, on 7 events each


## 3. The answer: nothing there

The announcement leg moves the affected wrapper **-14.1 bps** relative to its sibling. The effective leg moves it **+35.7 bps** — the *other way*. Neither is remotely unusual: we re-ran the same measurement on two thousand sets of **randomly chosen** dates, and swings this size show up by pure chance most of the time (*p* = 0.80 and 0.54).

We tried 9 different windows — one day, three, five, ten, twenty-one, and several straddling the announcement. The best of them still had a 42% chance of being luck.

> ⚠️ *How thin this is:* one of the eight announcement dates was originally transcribed wrong (2011-03-24 instead of 2011-04-05 for the 2011 Nasdaq-100 rebalance). Correcting it flipped that event from -120 to +36 bps and moved the pooled headline from -36.4 to -14.1 bps. When one typo can move your answer by 22 bps, you do not have a result.

## 4. The trap this study is really about

One window looked spectacular. Over the five days **before** the announcement, the affected wrapper lagged by -33.2 bps with a *t*-statistic of **-3.43** — the kind of number that gets a chart into a pitch deck.

It is an illusion. With only seven events, the *t*-statistic has to guess how variable these numbers are from seven data points, and those seven happened to land close together. The random-date test, which knows the true variability, prices exactly the same -33 bps at *p* = 0.45 — a coin flip.

> 🔬 *For the quants:* the cross-event *t* estimates its denominator from n = 7; the randomisation null for a 5-day CAR on this tape has a standard deviation several times the realised cross-event dispersion. Nine windows were examined; after Bonferroni every adjusted *p* is 1.000.

## 5. And the trade itself does not pay

Take the trade anyway: buy the affected wrapper, short the right amount of the sibling, hold ten days. Before any cost it returns **-7.7 bps** per event. After 5 bps of spread on each leg in and out, plus borrow on the short and financing on the bit that is not self-funding: **-29.3 bps**, winning 43% of the time. Every single cost/borrow combination we tried is negative, including the one where trading is free (-8.2 bps).

An investor who parked in T-bills and put this on at the 5 events that fall inside the T-bill era earned **+65 bps** over nineteen years — 50 live days out of 4,802, an excess-of-cash Sharpe of **+0.19** against SPY's +0.54. That is about 3 bps a year. And it is only positive because the T-bill era happens to start after the two events that lost the most.

## 6. Live check — the machinery does work (offline synthetic)

**This cell is synthetic, not the real tape.** We build a fake world with a genuine, planted shock after each announcement, and check the detector finds it; then a matched world with nothing planted, and check it stays quiet. If the detector passes both, the flat real-tape answer is about the market, not a bug.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from methodology_shock import data, strategy as st
planted = st.synthetic_detect(*data.synthetic_panel(signal_strength=1.0, seed=919)[:2],
                              window=(1, 10), n_draws=300)
null    = st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=919)[:2],
                              window=(1, 10), n_draws=300)
print('SYNTHETIC (not the real tape)')
print('  planted a 250 bps shock -> detector reports %+.0f bps, p = %.3f  (must fire)'
      % (planted['mean_car_bps'], planted['placebo_p']))
print('  planted nothing         -> detector reports %+.0f bps, p = %.3f  (must not fire)'
      % (null['mean_car_bps'], null['placebo_p']))

SYNTHETIC (not the real tape)
  planted a 250 bps shock -> detector reports +315 bps, p = 0.000  (must fire)
  planted nothing         -> detector reports +65 bps, p = 0.270  (must not fire)


## 7. What we can and cannot claim

Honesty about power: with seven events, the smallest shock this design could ever have called real is about **114 bps**. A genuine 20-40 bps footprint — which is roughly what the mechanism should produce once a basket-wide reweighting is spread across hundreds of names — would have been invisible to us. So the finding is not 'index rule changes do nothing'. It is the narrower, tradable statement: **nothing large enough to pay for the spread shows up in the wrapper you can buy.**

## Verdict

- **Signal — None.** Announcement leg -14.1 bps (*p* = 0.802), effective leg +35.7 bps (*p* = 0.541) — opposite signs, both indistinguishable from randomly chosen dates. All 9 windows have a Bonferroni-adjusted *p* of 1.000; both bootstrap CIs straddle zero; the two eras disagree.
- **Tradability — Mirage.** The hedged trade is negative **before** costs (-7.7 bps) and worse after (-29.3 bps). Every cell of the cost sweep is negative. The only version that looks profitable is the unhedged one, and its +24.3 bps is market beta wearing a costume.

---

*Every real-tape number above is frozen from [`docs/results.md`](../docs/results.md) (Fingerprint `547814cd71c7`, as-of 2026-06-30). The only cell that computes anything live is the synthetic control, and it is labelled.*